# Practical 11: Transformers with Hugging Face

**Concept:** **Transformers** (the architecture behind BERT, GPT and similar models) process all words in a sentence at once using **self-attention**, instead of reading left-to-right like RNNs. Training them from scratch is expensive, so in practice we use **pretrained models** and apply them out-of-the-box.

**Hugging Face Transformers** gives us these models behind one consistent API. In this practical we:
1. Run **sentiment analysis** through a ready-made `pipeline`.
2. Open the hood and use a **tokenizer + model** manually (tokenize → forward pass → softmax).
3. Do **text generation** with a pretrained GPT-style model.


## Install Transformers
`transformers` works on top of PyTorch, which was installed in Practical 10.


In [ ]:
!pip install transformers


## Import Required Libraries


In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

import torch
import torch.nn.functional as F

print("Transformers import successful")


## Pipeline 1: Sentiment Analysis
The easiest way to use a pretrained model is a **pipeline**: it bundles the tokenizer, the model and the label decoding for us. Without specifying a model, it downloads a tiny DistilBERT fine-tuned on movie reviews.


In [ ]:
sentiment_pipeline = pipeline("sentiment-analysis")

sentences = [
    "I absolutely loved this movie, it was fantastic!",
    "The service was slow and the food was terrible.",
    "This is the best day of my life!",
]

print("Sentiment Analysis")
print("-" * 50)
for sentence in sentences:
    result = sentiment_pipeline(sentence)[0]
    print(f"Text  : {sentence}")
    print(f"Label : {result['label']}  (confidence: {result['score']:.3f})")
    print()


## Under the Hood: The Tokenizer
A Transformer does not read words — it reads **sub-word tokens**. The tokenizer converts text into `input_ids` (token indices) and an `attention_mask` (which positions are real words vs padding). We decode the ids back to show the actual tokens.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

sentence = "The movie was simply amazing!"
tokens = tokenizer(sentence)

print("Tokenizer Output")
print("-" * 50)
print("input_ids      :", tokens["input_ids"])
print("attention_mask :", tokens["attention_mask"])
print("decoded tokens :", tokenizer.convert_ids_to_tokens(tokens["input_ids"]))
print("decoded text   :", tokenizer.decode(tokens["input_ids"]))


## Under the Hood: The Model Forward Pass
Now we use the model directly: encode the sentence into a batch, run a **forward pass** (no gradient needed) and turn the logits into probabilities with `softmax`.


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

inputs = tokenizer(sentence, return_tensors="pt")

print("Model Input Shapes")
print("-" * 50)
print("input_ids shape      :", inputs["input_ids"].shape)
print("attention_mask shape :", inputs["attention_mask"].shape)

with torch.no_grad():
    logits = model(**inputs).logits

probabilities = F.softmax(logits, dim=1)[0]
predicted = torch.argmax(probabilities).item()
label = model.config.id2label[predicted]

print()
print("Model Output")
print("-" * 50)
print("logits     :", logits)
print("probs      :", probabilities.tolist())
print("predicted  :", label, f"(confidence: {probabilities[predicted]:.3f})")


## Pipeline 2: Text Generation
`text-generation` uses a causal (GPT-style) language model that predicts the next word given the words so far. We generate a short continuation of a prompt. Note: the output can vary each run because generation is **sampling-based**.


In [ ]:
from transformers import pipeline as gen_pipeline

generator = gen_pipeline("text-generation", model="distilgpt2")

prompt = "Artificial intelligence is"
outputs = generator(prompt, max_new_tokens=20, do_sample=True, top_p=0.9, num_return_sequences=1)

print("Text Generation")
print("-" * 50)
print("Generated text:")
print(outputs[0]["generated_text"])


## Wrapping Up

Pretrained Transformers are applied the same way every time:
- **Tokenizer** turns raw text into `input_ids` + `attention_mask`.
- **Model** converts those ids into logits for the task (classify, generate, ...).
- The `pipeline` API bundles all of this into a single function call.

With just a few lines we used a state-of-the-art NLP architecture out-of-the-box — the same technique powers chatbots, search, translation and virtual assistants today.
